In [0]:
import zipfile, io, os

fecha = "2026-10-15"
zip_path = f"/Volumes/sepa/landing/raw/zips/sepa_{fecha}.zip"
destino = f"/Volumes/sepa/landing/raw/csv/fecha={fecha}"

ok, vacios, errores = [], [], []

with zipfile.ZipFile(zip_path) as outer:
    for nombre in outer.namelist():
        if not nombre.endswith(".zip"):
            continue
        comercio = os.path.basename(nombre).split("_")[2]   # ej: comercio-sepa-3
        datos = outer.read(nombre)
        if len(datos) == 0:
            vacios.append(comercio)
            continue
        try:
            carpeta = f"{destino}/comercio={comercio}"
            os.makedirs(carpeta, exist_ok=True)
            with zipfile.ZipFile(io.BytesIO(datos)) as inner:
                inner.extractall(carpeta)
            ok.append(comercio)
        except zipfile.BadZipFile:
            errores.append(comercio)

print(f"OK ({len(ok)}): {ok}")
print(f"Vacíos ({len(vacios)}): {vacios}")
print(f"Con error ({len(errores)}): {errores}")

In [0]:
import os, glob

def reparar_csv(ruta):
    """Une las filas que vienen partidas en varias lineas.
    Devuelve cuantas filas tuvo que reparar."""
    temporal = ruta + ".tmp"
    reparadas = 0
    with open(ruta, encoding="utf-8", newline="") as entrada, \
         open(temporal, "w", encoding="utf-8", newline="") as salida:
        encabezado = entrada.readline().rstrip("\r\n")
        salida.write(encabezado + "\n")
        separadores = encabezado.count("|")
        buffer, partes = "", 0
        for linea in entrada:
            linea = linea.rstrip("\r\n")
            if buffer == "" and "|" not in linea:
                salida.write(linea + "\n")
                continue
            buffer += linea
            partes += 1
            if buffer.count("|") >= separadores:
                salida.write(buffer + "\n")
                if partes > 1:
                    reparadas += 1
                buffer, partes = "", 0
        if buffer:
            salida.write(buffer + "\n")
    os.replace(temporal, ruta)
    return reparadas


fecha = "2026-10-15"
archivos = sorted(glob.glob(f"/Volumes/sepa/landing/raw/csv/fecha={fecha}/*/*.csv"))

total = 0
for ruta in archivos:
    n = reparar_csv(ruta)
    total += n
    if n > 0:
        print(f"{n:>8} filas reparadas en {ruta.split('csv/')[1]}")

print(f"Listo: {len(archivos)} archivos revisados, {total} filas reparadas en total")